# Avaliação Final do Modelo
Neste notebook, vamos carregar os melhor modelo salvo nas pastas `model`bem como seu respectivo normalizador (`scaler.pkl`).

In [1]:
import os
import h5py
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import load_model

I0000 00:00:1789852332.106959   49224 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789852332.134760   49224 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Carregando os Dados de Teste

In [2]:
# Se estiver usando o Google Colab, ajuste o caminho do arquivo H5

# Para execução local:
filename = '../data/N-CMAPSS_DS02-006.h5'

if not os.path.exists(filename):
    # Fallback caso esteja rodando de outra pasta
    filename = 'data/N-CMAPSS_DS02-006.h5'

with h5py.File(filename, 'r') as hdf:
    W_test = np.array(hdf.get('W_test'))
    X_s_test = np.array(hdf.get('X_s_test'))
    Y_test = np.array(hdf.get('Y_test'))
    A_test = np.array(hdf.get('A_test'))

def create_temporal_features_safe(W, X_s, Y, A, window):
    matriz_base = np.concatenate((W, X_s), axis=1).astype('float32')
    df = pd.DataFrame(matriz_base)
    df['unit'] = A[:, 0].astype('float32')
    df['RUL'] = Y.flatten().astype('float32')
    
    df_mean = df.groupby('unit').rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True).sort_index().astype('float32')
    df_var = df.groupby('unit').rolling(window=window, min_periods=1).var().fillna(0).reset_index(level=0, drop=True).sort_index().astype('float32')
    
    df_mean = df_mean[df.columns[:-2]]
    df_var = df_var[df.columns[:-2]]
    df_raw = pd.DataFrame(matriz_base)
    
    X_temporal = pd.concat([df_raw, df_mean, df_var], axis=1).values.astype('float32')
    y_labels = df['RUL'].values.astype('float32')
    
    del df, df_mean, df_var, df_raw, matriz_base
    gc.collect()
    
    return X_temporal, y_labels

## 2. Definindo a Configuração dos Modelos

In [3]:
# Configuração dos modelos a serem avaliados
# A melhor janela obtida sendo a 20
model_config = [
    {"name": "Melhor Modelo ", "folder": "model", "window": 20}]


resultados = []

for config in model_config:
    print(f"\n {'='*50}\n Carregando {config['name']} da pasta '{config['folder']}'...")
    
    # Caminhos dos arquivos
    model_path = os.path.join(config['folder'], 'model.keras')
    scaler_path = os.path.join(config['folder'], 'scaler.pkl')
    
    if not os.path.exists(model_path) or not os.path.exists(scaler_path):
        print(f"ERRO: Arquivos não encontrados para {config['name']}! Padrão esperado: model.keras e scaler.pkl")
        continue
        
    # Carregando Modelo e Normalizador
    model = load_model(model_path)
    scaler = joblib.load(scaler_path)
    
    # Gerando as features de teste com a janela específica deste modelo
    print(f"Gerando matriz temporal (Janela = {config['window']})...")
    X_test_raw, y_test_labels = create_temporal_features_safe(W_test, X_s_test, Y_test, A_test, window=config['window'])
    
    print("Normalizando e Prevendo...")
    X_test_scaled = scaler.transform(X_test_raw).astype('float32')
    
    y_pred = model.predict(X_test_scaled, verbose=0)
    
    # Calculando Métricas
    mse = mean_squared_error(y_test_labels, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_labels, y_pred)
    r2 = r2_score(y_test_labels, y_pred)
    
    print(f"Métricas {config['name']}:")
    print(f"  - R²   : {r2:.4f}")
    print(f"  - MAE  : {mae:.2f}")
    print(f"  - RMSE : {rmse:.2f}")
    print(f"  - MSE  : {mse:.2f}")
    
    resultados.append({
        'Modelo': config['name'],
        'Janela': config['window'],
        'R²': r2,
        'MAE': mae,
        'RMSE': rmse,
        'MSE': mse
    })
    
    # Limpeza de Memória
    del model, scaler, X_test_raw, X_test_scaled, y_test_labels, y_pred
    tf.keras.backend.clear_session()
    gc.collect()


 Carregando Melhor Modelo  da pasta 'model'...


I0000 00:00:1789852334.300899   49224 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5743 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a
/home/anderson/miniconda3/envs/intelligent_systems/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Gerando matriz temporal (Janela = 20)...
Normalizando e Prevendo...


I0000 00:00:1789852338.303637   49307 service.cc:153] XLA service 0x7a9dc402f680 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789852338.303687   49307 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5050 Laptop GPU, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.9.0; DNN: 9.17.0)
I0000 00:00:1789852338.338309   49307 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789852338.444601   49307 cuda_dnn.cc:461] Loaded cuDNN version 91700
I0000 00:00:1789852339.559718   49307 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Métricas Melhor Modelo :
  - R²   : 0.8642
  - MAE  : 5.16
  - RMSE : 6.99
  - MSE  : 48.85


## 3. Tabela Comparativa Final

In [4]:
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values(by='R²', ascending=False).reset_index(drop=True)

# Arredondando os valores
df_resultados['R²'] = df_resultados['R²'].apply(lambda x: f"{x:.4f}")
df_resultados['MAE'] = df_resultados['MAE'].apply(lambda x: f"{x:.2f}")
df_resultados['RMSE'] = df_resultados['RMSE'].apply(lambda x: f"{x:.2f}")
df_resultados['MSE'] = df_resultados['MSE'].apply(lambda x: f"{x:.2f}")

print("\nRANKING FINAL DOS MODELOS\n ")

# Imprimindo como uma tabela Markdown estruturada
from IPython.display import display, Markdown

markdown_table = "| Modelo | Janela | R² Score | MAE | RMSE | MSE |\n "
markdown_table += "| :--- | :---: | :---: | :---: | :---: | :---: |\n "
for index, row in df_resultados.iterrows():
    markdown_table += f"| **{row['Modelo']}** | {row['Janela']} | {row['R²']} | {row['MAE']} | {row['RMSE']} | {row['MSE']} |\n"

display(Markdown(markdown_table))


RANKING FINAL DOS MODELOS
 


| Modelo | Janela | R² Score | MAE | RMSE | MSE |
 | :--- | :---: | :---: | :---: | :---: | :---: |
 | **Melhor Modelo ** | 20 | 0.8642 | 5.16 | 6.99 | 48.85 |
